# 10.2 Coding Practice: Exploring BERT-Family Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/mynewbook/blob/master/module10/week10_bert_practice.ipynb)

This guided practice follows the Week 10 reading, *Transformers, Pretrained Models, and Transfer Learning*. You will not train anything. Instead, you will open up pretrained models and see the ideas from the reading in action:

1. how a BERT-family model splits text into **subword tokens**
2. how BERT plays its **fill-in-the-blank** game (masked language modeling)
3. how the same word gets **different vectors in different contexts**
4. how a **ready-made sentiment classifier** handles tricky sentences
5. how to read a **model card** before trusting a model

We will use **DistilBERT**, a smaller and faster version of BERT that runs comfortably on a free Colab CPU, and a ready-made sentiment model.

Each part follows the same rhythm: a short explanation, a question to **predict first**, a small piece of code to run or complete, and a look at the output. Plan for about 40 to 50 minutes.

This is ungraded practice. There is nothing to submit. If you get stuck, look at the hints in the code comments, or open the solution notebook.

## Learning goals

- Inspect subword tokens and special tokens such as `[CLS]`, `[SEP]`, and `[MASK]`.
- Use a masked language model and explain how context on both sides changes its guesses.
- Compare contextual vectors for the same word in different sentences.
- Use a ready-made fine-tuned classifier and judge its predictions on challenging sentences.
- Read a model card and decide whether a model fits a task.

## Setup

Choose a standard **CPU** runtime in Colab (Runtime > Change runtime type > CPU). Run the cells from top to bottom.

The first run downloads two public models from the Hugging Face Hub (together roughly 750 MB), which can take a minute or two. No Hugging Face account, token, or API key is needed. If Colab shows a message about an `HF_TOKEN`, you can ignore it.

The next cell installs the tested version of the Transformers library. Colab already includes PyTorch. If Colab asks you to restart the runtime, restart it and run all cells again.

In [ ]:
%pip -q install "transformers==5.17.0"

In [ ]:
import torch
import transformers
from transformers import AutoModel, AutoTokenizer, pipeline
from huggingface_hub import ModelCard

transformers.logging.set_verbosity_error()  # hide long loading messages
transformers.logging.disable_progress_bar()  # hide progress bars
torch.set_num_threads(2)
print("Transformers:", transformers.__version__, "| PyTorch:", torch.__version__)

# The pretrained BERT-family model we explore in Parts 1 to 3.
BASE_MODEL = "distilbert/distilbert-base-uncased"
review = "Everyone told me the film was boring, but I loved it."

## 1. Subword tokens (about 8 minutes)

Before a model can read text, a **tokenizer** splits it into tokens and turns each token into an ID number. Every model has its **own** tokenizer, and you must always use the tokenizer that matches the model.

BERT-family tokenizers use **subwords**: common words stay whole, while rare or long words are split into smaller pieces. A piece that starts with `##` continues the previous piece.

**Predict first:** our review has 11 words. Will the tokenizer produce more or fewer than 11 tokens? Which extra tokens do you expect?

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

encoded = tokenizer(review)
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
print("Number of tokens:", len(tokens))
print("Token          ID")
for token, token_id in zip(tokens, encoded["input_ids"]):
    print(f"{token:14s} {token_id}")

Notice three things:

- The tokenizer lowercases the text, because this is an *uncased* model.
- Punctuation gets its own tokens.
- `[CLS]` is added at the start and `[SEP]` at the end. In the reading, BERT uses the vector for `[CLS]` as a summary of the whole text when it classifies. Remember that this is a BERT-style convention: other models summarize a text in other ways.

### TODO 1: Find a word that gets split

Write a sentence with at least one long or unusual word (a movie title, a technical term, or a made-up word) and look at its tokens.

In [ ]:
# TODO 1: replace the sentence below with your own.
my_sentence = "Write your own sentence here."

# This line is complete: it shows how the tokenizer splits your sentence.
print(tokenizer.tokenize(my_sentence))

**Question 1:** Which words were split into pieces? Why is it useful that the tokenizer can split an unknown word into pieces instead of replacing it with an "unknown" token, as our Week 9 LSTM vocabulary did?

## 2. BERT's fill-in-the-blank game (about 8 minutes)

The reading explained that BERT is pretrained by **masked language modeling**: some words are hidden behind `[MASK]`, and the model guesses them using the words on **both** sides.

The `fill-mask` pipeline lets you play this game with DistilBERT. It returns the five most likely words for the blank, each with a probability.

**Predict first:** in *"Everyone told me the film was [MASK], but I loved it."*, what kind of word do you expect? Does the part **after** the blank (*but I loved it*) help?

In [ ]:
fill_mask = pipeline(
    "fill-mask",
    model=BASE_MODEL,
    tokenizer=BASE_MODEL,
    device=-1,  # CPU
    model_kwargs={"disable_mmap": True},  # load weights into memory for CPU compatibility
)


def show_guesses(sentence, k=5):
    print(sentence)
    for guess in fill_mask(sentence, top_k=k):
        print(f"   {guess['token_str']:12s} {guess['score']:.3f}")
    print()


show_guesses("Everyone told me the film was [MASK], but I loved it.")
show_guesses("Everyone told me the film was [MASK].")

The only difference between the two sentences is the text **after** the blank. A left-to-right model reading *"Everyone told me the film was ..."* could not use that text at all, but BERT can.

### TODO 2: Change the right-hand context

Write a sentence with one `[MASK]`, where the words **after** the blank should change the guess. Then make a second version with a different ending. For example, try endings that suggest a positive or a negative opinion.

In [ ]:
# TODO 2: write two sentences that differ only AFTER the [MASK].
# Each sentence must contain exactly one [MASK].
sentence_a = "The ending was [MASK], so I ..."
sentence_b = "The ending was [MASK], so I ..."

show_guesses(sentence_a)
show_guesses(sentence_b)

**Question 2:** Did the guesses change in the way you expected? Look at the probabilities: is the model very sure about any guess? What does that tell you about how many words could reasonably fill the blank?

## 3. Same word, different vectors (about 10 minutes)

In Module 3, a static embedding such as word2vec gave each word **one** vector. The reading explained that a transformer gives each word a **contextual** vector that depends on the sentence.

We will test this with the word *score*, which can mean music or a rating:

| Sentence | Meaning of *score* |
| --- | --- |
| The film's score was haunting. | music |
| The film's score on the review site was low. | rating |
| The soundtrack and the score were beautiful. | music |
| Critics gave the film a low score. | rating |

First, notice that the tokenizer gives *score* the **same ID** in every sentence. Any difference must therefore come from the context.

In [ ]:
score_sentences = [
    "The film's score was haunting.",
    "The film's score on the review site was low.",
    "The soundtrack and the score were beautiful.",
    "Critics gave the film a low score.",
]
for sentence in score_sentences:
    print(tokenizer.convert_tokens_to_ids("score"), "|", sentence)

Now load the DistilBERT model itself (the pretrained **body** from the reading) and pull out the vector it builds for one word in a sentence. The helper function below is complete. It returns a vector of 768 numbers.

In [ ]:
model = AutoModel.from_pretrained(BASE_MODEL, disable_mmap=True)
model.eval()


def word_vector(sentence, word):
    """Return DistilBERT's contextual vector for `word` in `sentence`."""
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.inference_mode():
        outputs = model(**inputs)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    position = tokens.index(word)  # where the word sits in this sentence
    return outputs.last_hidden_state[0, position]


v = word_vector(score_sentences[0], "score")
print("Vector length:", v.shape[0])
print("First five numbers:", [round(x, 3) for x in v[:5].tolist()])

To compare two vectors we use **cosine similarity**, which you met in Week 4: values near 1 mean the vectors point in a very similar direction.

**Predict first:** which pairs of sentences should give the most similar vectors for *score*?

### TODO 3: Build the similarity table

Complete the loop so it prints the cosine similarity between the *score* vector of every pair of sentences.

In [ ]:
score_vectors = [word_vector(s, "score") for s in score_sentences]

print("      S1     S2     S3     S4")
for i, vec_i in enumerate(score_vectors):
    row = []
    for vec_j in score_vectors:
        # TODO 3: compute the cosine similarity between vec_i and vec_j.
        # Hint:  torch.nn.functional.cosine_similarity(vec_i, vec_j, dim=0).item()
        similarity = ...
        row.append(similarity)
    print(f"S{i + 1} " + " ".join(f"{s:6.3f}" for s in row))

**Question 3:** Which pairs are most similar? Do the two "music" sentences (S1 and S3) and the two "rating" sentences (S2 and S4) group together? Why would a static embedding give 1.000 for every pair?

## 4. A ready-made sentiment classifier (about 10 minutes)

In the reading, the first way to use a pretrained model was to **use a ready-made model** that someone already fine-tuned. Here we load one: a RoBERTa model (a member of the BERT family) fine-tuned for sentiment, with **three** labels: negative, neutral, and positive.

This is a different model from DistilBERT, so it has its own tokenizer. Look at how differently it splits the same words: `Ġ` marks a token that starts with a space.

In [ ]:
SENTIMENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
classifier = pipeline(
    "sentiment-analysis",
    model=SENTIMENT_MODEL,
    tokenizer=SENTIMENT_MODEL,
    device=-1,
    model_kwargs={"disable_mmap": True},
)
print(classifier.tokenizer.tokenize(review))

**Predict first:** for each sentence below, write down the label you would give (negative, neutral, or positive) **before** running the cell. Which ones do you think the model will get wrong?

| # | Sentence | Challenge |
| --- | --- | --- |
| 1 | Everyone told me the film was boring, but I loved it. | contrast |
| 2 | The film was not bad. | negation |
| 3 | The film was not good. | negation |
| 4 | The film was neither good nor bad. | mixed / neutral |
| 5 | Oh great, another three-hour movie with no plot. | sarcasm |
| 6 | The movie starts at 7 pm. | no opinion |
| 7 | I expected to hate it, but it was fun. | contrast |
| 8 | The acting was great, but the story made no sense. | mixed |

In [ ]:
challenge_sentences = [
    "Everyone told me the film was boring, but I loved it.",
    "The film was not bad.",
    "The film was not good.",
    "The film was neither good nor bad.",
    "Oh great, another three-hour movie with no plot.",
    "The movie starts at 7 pm.",
    "I expected to hate it, but it was fun.",
    "The acting was great, but the story made no sense.",
]


def show_predictions(sentences):
    print("negative  neutral  positive  label     | sentence")
    for sentence, scores in zip(sentences, classifier(sentences, top_k=None)):
        by_label = {item["label"]: item["score"] for item in scores}
        best = max(by_label, key=by_label.get)
        print(f"{by_label['negative']:.3f}     {by_label['neutral']:.3f}    "
              f"{by_label['positive']:.3f}     {best:9s} | {sentence}")


show_predictions(challenge_sentences)

### TODO 4: Try to fool the model

Write at least three sentences of your own that you think will be hard for the model: for example sarcasm, double negation, a mixed review, or a review about a topic far from social media and movies.

In [ ]:
# TODO 4: add at least three challenging sentences of your own.
my_sentences = [
    "Write a challenging sentence here.",
]

show_predictions(my_sentences)

**Question 4:** Which predictions do you disagree with? Pick one and explain why the model might have made that mistake. Does a high score (for example 0.80) mean the prediction is correct?

## 5. Read the model cards (about 8 minutes)

The reading said to **read the model card** before trusting a model. A model card is the model's user guide: training data, labels, intended uses, and limitations.

The `ModelCard` tool downloads a card as text. We will compare two cards:

- the sentiment model from Part 4
- the DistilBERT movie-review sentiment model you used in Week 9

First, look at the short summary (metadata) at the top of each card.

In [ ]:
WEEK9_MODEL = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
cards = {name: ModelCard.load(name) for name in [SENTIMENT_MODEL, WEEK9_MODEL]}


def card_headings(card):
    """List the section headings of a model card, skipping code examples."""
    headings, in_code = [], False
    for line in card.text.splitlines():
        if line.strip().startswith("```"):
            in_code = not in_code
        elif line.startswith("#") and not in_code:
            headings.append(line)
    return headings


for name, card in cards.items():
    info = card.data.to_dict()
    print(name)
    print("   datasets:", info.get("datasets"))
    print("   language:", info.get("language"))
    print("   license: ", info.get("license"))
    print("   sections:", card_headings(card)[:12])
    print()

The Week 9 model card has a section called **Risks, Limitations and Biases**. The helper below prints any section of a card by its heading.

In [ ]:
def show_section(card, heading, max_chars=1500):
    """Print the text under a heading in a model card."""
    lines = card.text.splitlines()
    start = next(i for i, line in enumerate(lines) if heading in line and line.startswith("#"))
    section = []
    for line in lines[start + 1:]:
        if line.startswith("# ") or line.startswith("## "):
            break
        section.append(line)
    print("\n".join(section).strip()[:max_chars])


show_section(cards[WEEK9_MODEL], "Risks, Limitations and Biases")

Now print the opening description of the Part 4 sentiment model's card.

In [ ]:
print(cards[SENTIMENT_MODEL].text.strip()[:900])

### TODO 5: Fill in a model card checklist

Using what you just read, fill in the checklist for the **Part 4 sentiment model**. Write short answers in plain words.

In [ ]:
# TODO 5: replace each "..." with a short answer.
checklist = {
    "training data": "...",
    "labels": "...",
    "language": "...",
    "limitations described on the card?": "...",
    "a task it fits well": "...",
    "a task I would test carefully first": "...",
}

for question, answer in checklist.items():
    print(f"{question:38s} {answer}")

**Question 5:** The Week 9 model was fine-tuned on movie review sentences, and the Part 4 model on tweets. A local hospital wants to sort patient comments into positive, neutral, and negative. Which model would you try first, and what would you check before trusting it?

## Wrap-up

### Reflection

1. Which of the three ways to use a pretrained model (ready-made, feature extractor, fine-tuning) did you use in Part 4? Which part of this notebook used the pretrained **body** directly?
2. In Part 2, the guesses changed when you changed words **after** the blank. Why could a left-to-right model like GPT not do this?
3. Name one thing you would always check before using a pretrained model for a real project.

### Looking ahead

Today you used pretrained models without training anything. The same DistilBERT body you explored in Part 3 can become a classifier for **your** data: either by training a small classifier on its vectors, or by fine-tuning it. Many modern chat assistants work the other way around: they are large **decoder** models that you use by writing instructions (prompts) rather than by training.

## Validation checks

These checks confirm that the notebook ran from top to bottom. If one fails, rerun the notebook from the start (Runtime > Restart and run all in Colab).

In [ ]:
assert tokens[0] == "[CLS]" and tokens[-1] == "[SEP]"
assert len(fill_mask("The film was [MASK].", top_k=5)) == 5
assert len(score_vectors) == 4 and score_vectors[0].shape[0] == 768
same = torch.nn.functional.cosine_similarity(score_vectors[0], score_vectors[0], dim=0).item()
assert abs(same - 1.0) < 1e-4
labels = {item["label"] for item in classifier(review, top_k=None)}
assert labels == {"negative", "neutral", "positive"}
assert "Risks, Limitations and Biases" in cards[WEEK9_MODEL].text
print("All checks passed.")